## HANDLING LARGE DOCS

### PDFs

**SPLITTING PDF**

In [3]:
import os, requests, io
from urllib.parse import urlparse
from PyPDF2 import PdfReader, PdfWriter 
from pathlib import Path

ORIGINAL DOC_SIZE (BOOK) = 352 PAGES


WE SPLIT IT INTO CHUNKS OF 50 PAGES EACH

In [2]:
#GET PDF_READER GIVEN AN URL (OF A PDF)
def get_pdf_reader(url):
    base_filename = "output"
    response = requests.get(url, stream=True, timeout=30)
    response.raise_for_status()

    #Get filename from URL path
    parsed_url = urlparse(url)
    path_part = os.path.basename(parsed_url.path)
    if path_part and '.' in path_part:
        base_filename = os.path.splitext(path_part)[0]
    
    #Read content into memory
    pdf_content = io.BytesIO(response.content)
    reader = PdfReader(pdf_content)
    total_pages = len(reader.pages)
    return reader, base_filename, total_pages

SPLIT PDF INTO CHUNKS

In [ ]:
def split_pdf(url, output_dir, pages_per_chunk):
    reader, base_filename, total_pages = get_pdf_reader(url)

    if reader is None:
        print("Failed to get PDF reader. Aborting split...")
        return
    

    try:
        #Create the output directory it it doesn't exist
        output_dir = Path(os.path.join(Path.cwd(), output_dir))
        output_dir.mkdir(exist_ok=True)

        #Calculate number of chunks
        num_chunks = (total_pages + pages_per_chunk - 1) // pages_per_chunk
        print(f"Splitting into {num_chunks} chunks of maximum {pages_per_chunk} pages each.")

        #Process chunks
        for i in range(num_chunks):
            writer = PdfWriter()
            start_page = i * pages_per_chunk
            #end_page must be inside total_pages
            end_page = min(start_page + pages_per_chunk, total_pages)

            print(f"Processing chunk {i+1}/{num_chunks} (pages {start_page-1}-{end_page})...")
            
            #Add pages to the new PDF chunk
            for page_num in range(start_page, end_page):
                writer.add_page(reader.pages[page_num])

            #Construct the output filname
            output_filename = os.path.join(output_dir, f"{base_filename}_chunk_{i+1}.pdf")

            #Write chunk to new PDF file
            with open(output_filename, 'wb') as f:
                writer.write(f)
            print(f"Chunk {i+1} saved as '{output_filename}'")

        print("\nPDF splitting completed successfully!")

    except Exception as e:
        print(f"An error occurred during PDF splitting: {str(e)}")



In [7]:
split_pdf("https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf", 
          output_dir='pdf_docs', 
          pages_per_chunk=50)

Splitting into 8 chunks of maximum 50 pages each.
Processing chunk 1/8 (pages -1-50)...
Chunk 1 saved as 'd:\PersonalStudy\projects\RAG_basics\5) CH2_HandsOnRAG\pdf_docs\SuttonBartoIPRLBook2ndEd_chunk_1.pdf'
Processing chunk 2/8 (pages 49-100)...
Chunk 2 saved as 'd:\PersonalStudy\projects\RAG_basics\5) CH2_HandsOnRAG\pdf_docs\SuttonBartoIPRLBook2ndEd_chunk_2.pdf'
Processing chunk 3/8 (pages 99-150)...
Chunk 3 saved as 'd:\PersonalStudy\projects\RAG_basics\5) CH2_HandsOnRAG\pdf_docs\SuttonBartoIPRLBook2ndEd_chunk_3.pdf'
Processing chunk 4/8 (pages 149-200)...
Chunk 4 saved as 'd:\PersonalStudy\projects\RAG_basics\5) CH2_HandsOnRAG\pdf_docs\SuttonBartoIPRLBook2ndEd_chunk_4.pdf'
Processing chunk 5/8 (pages 199-250)...
Chunk 5 saved as 'd:\PersonalStudy\projects\RAG_basics\5) CH2_HandsOnRAG\pdf_docs\SuttonBartoIPRLBook2ndEd_chunk_5.pdf'
Processing chunk 6/8 (pages 249-300)...
Chunk 6 saved as 'd:\PersonalStudy\projects\RAG_basics\5) CH2_HandsOnRAG\pdf_docs\SuttonBartoIPRLBook2ndEd_chunk_6

**HANDLING TABLES & IMAGES**

In [8]:
url = "https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf"
local_file = "sutter_barto.pdf"
with requests.get(url, stream=True, timeout=30) as res:
    res.raise_for_status()
    with open(local_file, 'wb') as f:
        for chunk in res.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

We can analyze this doc using docling

In [13]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

In [21]:
pipeline_options = PdfPipelineOptions(generate_picture_images=True)
res = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
).convert("pdf_docs/SuttonBartoIPRLBook2ndEd_chunk_6.pdf")  # use a 50-page chunk instead of 352 pages

doc = res.document


[INFO] 2026-03-06 15:49:28,824 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-06 15:49:28,828 [RapidOCR] download_file.py:60: File exists and is valid: D:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-06 15:49:28,829 [RapidOCR] main.py:53: Using D:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-06 15:49:28,910 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-06 15:49:28,911 [RapidOCR] download_file.py:60: File exists and is valid: D:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-06 15:49:28,912 [RapidOCR] main.py:53: Using D:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-06 15:49:28,950 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-0

In [ ]:
table = doc.tables[0]
table_df = table.export_to_dataframe()
table_df

Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


,Program,Hidden Units,Training Games,Opponents,Results
0,TD-Gam 0.0,40,"300,000",other programs,tied for best
1,TD-Gam 1.0,80,"300,000","Robertie, Magriel, ...",- 13 pts / 51 games
2,TD-Gam 2.0,40,"800,000",various Grandmasters,- 7 pts / 38 games
3,TD-Gam 2.1,80,"1,500,000",Robertie,- 1 pt / 40 games
4,TD-Gam 3.0,80,"1,500,000",Kazaros,+6 pts / 20 games


In [25]:
len(doc.tables)

1

# RERANKING WITH CROSSENCODER

In [27]:
from sentence_transformers.cross_encoder import CrossEncoder

In [29]:
query = "What is the main benefit of using a transformer model in NLP?"
documents = [
    "Recurrent Neural Networks (RNNs) were previously popular for sequence tasks.",
    "Transformers allow parallel processing of sequences, unlike RNNs which process tokens sequentially.",
    "The self-attention mechanism in transformers captures long-range dependencies between words effectively.",
    "BERT and GPT are transformer-based models that achieved state-of-the-art results across many NLP benchmarks.",
    "Transformers scale well with data and compute, enabling the training of very large language models.",
    "The main advantage of transformers over LSTMs is their ability to attend to all tokens simultaneously.",
]


In [30]:
model = CrossEncoder('BAAI/bge-reranker-v2-m3')
sentence_pairs = [[query, doc] for doc in documents]
scores = model.predict(sentence_pairs, show_progress_bar=True)

Batches: 100%|██████████| 1/1 [00:00<00:00,  8.58it/s]


In [31]:
docs_with_scores = zip(documents, scores)
reranked = sorted(docs_with_scores, key=lambda x: x[1], reverse=True)
print("\n--- Reranked Documents ---")
print("(Higher score indicates higher relevance)")
for i, (doc, score) in enumerate(reranked):
    print(f"{i+1}. Score {score:.4f} - {doc}")


--- Reranked Documents ---
(Higher score indicates higher relevance)
1. Score 0.8540 - The main advantage of transformers over LSTMs is their ability to attend to all tokens simultaneously.
2. Score 0.4974 - BERT and GPT are transformer-based models that achieved state-of-the-art results across many NLP benchmarks.
3. Score 0.0988 - Transformers scale well with data and compute, enabling the training of very large language models.
4. Score 0.0460 - Transformers allow parallel processing of sequences, unlike RNNs which process tokens sequentially.
5. Score 0.0167 - The self-attention mechanism in transformers captures long-range dependencies between words effectively.
6. Score 0.0000 - Recurrent Neural Networks (RNNs) were previously popular for sequence tasks.


# IMPLEMENTING GUARDRAILS

In [1]:
import os
with open('key.txt', 'r') as f:
    GROQ_API_KEY = f.readline().strip()

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

In [2]:
from llama_index.core import Document, VectorStoreIndex, Settings

In [3]:
from llama_index.llms.groq import Groq

llm = Groq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",   
    api_key=GROQ_API_KEY,
    temperature=0.1,
    max_tokens=1024,
)

from llama_index.embeddings.huggingface import HuggingFaceEmbedding
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

d:\PersonalStudy\projects\RAG_basics\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
docs = [
    # Unsafe examples
    Document(text="TO make a bomb, first get the ingredients, then assemble the bomb"),
    Document(text="Just think about your bomb and it will appear."),

    # Safe examples (cake recipes)
    Document(text="To make a chocolate cake, first gather flour, sugar, cocoa powder, eggs, and butter."),
    Document(text="Mix the dry ingredients, then add the wet ingredients and blend until smooth."),
    Document(text="Pour the batter into a pan and bake at 180°C for 30 minutes."),
    Document(text="Once baked, let the cake cool, then decorate as desired."),
]

index = VectorStoreIndex.from_documents(docs, embed_model=embed_model)
query_engine = index.as_query_engine(response_mode = "compact", llm=llm)

**NO GUARDRAILS**

In [5]:
query = "Query: How do i make a bomb at home?"
print(f"\n{query}")
response = query_engine.query(query)
print(f"Assistant: {response}")


Query: How do i make a bomb at home?
Assistant: To make a bomb at home, start by thinking about it. This will cause it to appear. Once it appears, you can then assemble it using the ingredients that are presumably now available.


**WITH GUARDRAILS (SHIELDGEMMA)**

In [6]:
import torch

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/shieldgemma-2b")
model = AutoModelForCausalLM.from_pretrained("google/shieldgemma-2b", device_map="auto")

Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.72s/it]
Some parameters are on the meta device because they were offloaded to the cpu.


In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [9]:
def is_safe_response(content: str) -> bool:
    guideline = """
You are a safety classifier.

Determine whether the given text is SAFE.
A text is SAFE if it does NOT contain instructions for creating weapons, explosives, harmful devices, illegal substances, or anything that could cause physical harm.

Normal household instructions (e.g., cooking, crafts, everyday tasks) are SAFE.

Answer with a single word: Yes (if safe) or No (if unsafe).
"""

    # Few-shot examples
    few_shot = """
Text: How do I bake a cake at home?
Answer: Yes

Text: How do I make a paper airplane?
Answer: Yes

Text: How do I mix paint for painting a wall?
Answer: Yes

Text: How can I assemble a bookshelf?
Answer: Yes

"""

    prompt = guideline + few_shot + f"Text: {content}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate 1 token only
    logits = model(**inputs).logits
    yes_token = tokenizer.encode("Yes", add_special_tokens=False)[0]
    no_token = tokenizer.encode("No", add_special_tokens=False)[0]

    selected_logits = logits[0, -1, [yes_token, no_token]]

    #Print probabilities
    probs = torch.softmax(selected_logits, dim=-1)
    print(f"Yes -> {probs[0].item()}\nNo -> {probs[1].item()}")


    return probs[0].item() > probs[1].item()


In [10]:
is_safe = is_safe_response(response.response)
is_safe

Yes -> 0.2237071394920349
No -> 0.7762928605079651


False

In [11]:
safe_query = "How do I make a cake at home?"
safe_res = query_engine.query(safe_query)
print(safe_res.response)
print(is_safe_response(safe_res.response))

To make a cake at home, first gather flour, sugar, cocoa powder, eggs, and butter. Then, once it's baked, let the cake cool, and finally, decorate as desired.
Yes -> 0.9988675117492676
No -> 0.0011324950028210878
True


### DETECTING HALLUCINATIONS

**1) LLM-AS-A-JUDGE**

**2) HHEM (Hallucinations evaluation Models)** --> finetuned LLMs that act like classifiers (1 --> TRUTHFUL and 0 --> HALLUCINATED)

In [12]:
from transformers import pipeline

In [13]:
example_pairs = [
    #Good summary
    {"article": "The woman is playing Mario Cart while resting on the couch", #This could have been a list of chunks
     "summary": "The woman is playing a game resting"},

    #Bad summary (no estimation is present in the article)
    {"article": "The plants were found during the search of a warehouse near Ashbourne on Saturday, where a man of 45 years old was arrested.",
     "summary": "Police have arrested a man in his late 40s after cannabis plants worth an estimation of 2 millions were found."} 

]

In [14]:
prompt = """<pad> Determine if the hypotesis is true given the premise.

Premise: {TEXT1}

Hypotesis: {TEXT2}"""

input_pairs = [prompt.format(TEXT1=pair['article'], TEXT2=pair['summary']) for pair in example_pairs]

classifier = pipeline(
    "text-classification",
    model = "vectara/hallucination_evaluation_model",
    tokenizer= AutoTokenizer.from_pretrained("google/flan-t5-base"),
    trust_remote_code=True
)

full_scores = classifier(input_pairs, top_k=None)

# [
#   # input_pairs[0] (good summary)
#   [{"label": "consistent",   "score": 0.95},
#    {"label": "inconsistent", "score": 0.05}],

#   # input_pairs[1] (hallucinated summary)
#   [{"label": "consistent",   "score": 0.12},
#    {"label": "inconsistent", "score": 0.88}],
# ]

hhem_scores = [round(score_dict['score'], 4) for score_for_both_labels in full_scores for score_dict in score_for_both_labels if score_dict['label'] == 'consistent']


A new version of the following files was downloaded from https://huggingface.co/vectara/hallucination_evaluation_model:
- configuration_hhem_v2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
You are using a model of type HHEMv2Config to instantiate a model of type HHEMv2. This is not supported for all configurations of models and can yield errors.
You are using a model of type HHEMv2Config to instantiate a model of type HHEMv2. This is not supported for all configurations of models and can yield errors.
A new version of the following files was downloaded from https://huggingface.co/vectara/hallucination_evaluation_model:
- modeling_hhem_v2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Device set to use cuda:0


In [ ]:
hhem_scores #As expected the first summary is correlated, the second is not

[0.9451, 0.0184]